<a href="https://colab.research.google.com/github/BaffoBello14/ipynbVideoTrans/blob/main/pyvideotrans_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# pyVideoTrans – Google Colab Edition

Traduci video da una lingua all'altra direttamente su Google Colab:
riconoscimento vocale (ASR) → traduzione sottotitoli → sintesi vocale (TTS) → video finale.

---

**Pipeline:**
```
Video input
  ↓  [Cella 4] ASR  – estrae audio → trascrive → SRT sorgente
  ↓  [Cella 5] Translate – SRT sorgente → SRT tradotto
  ↓  [Cella 6] TTS  – SRT tradotto → audio doppiato
  ↓  [Cella 7] Assembla – audio + video → MP4 finale
```
oppure esegui tutto in un colpo con la **Cella 6 – Pipeline completa**.

**Correzione traduzione:** nella **Cella 4 – Configurazione** imposta `PAUSE_AFTER_TRANSLATION = True`: dopo la traduzione la cella pipeline si ferma e chiede conferma in console; modifica il file `TARGET_SRT` (stesso `OUTPUT_DIR`, nome tipo `en.srt` se `TARGET_LANG` è `en`), salva, poi premi **Invio** nella finestra di input per continuare con TTS. Con `False` il run va dritto fino al MP4 (per correggere dopo, modifica l’SRT e rilancia solo la parte che ti serve, oppure usa `PAUSE…`).

**Altri flag** nella Cella 4: `CLEAR_CACHE_BEFORE_RUN` (se `True` svuota `OUTPUT_DIR` all’inizio: **perdi** correzioni manuali), `RUN_DIAGNOSTICS_AT_INIT` (alla fine della Cella 5 stampa provider e un campione di voci Edge-TTS).

---
> **Suggerimento GPU:** Menu **Runtime → Cambia tipo di runtime → GPU (T4)**  
> Velocizza drasticamente la trascrizione Whisper.


## Cella 1 – Installa ffmpeg e dipendenze di sistema

In [ ]:
import subprocess, sys

# ffmpeg è necessario per estrarre audio e assemblare il video finale
# ffmpeg da apt su Colab è spesso 4.x (senza -fps_mode). Il backend usa -vsync vfr come fallback.
# Per ffmpeg più recente: https://johnvansickle.com/ffmpeg/ (build statica) oppure conda-forge.
subprocess.run(["apt-get", "install", "-y", "-q", "ffmpeg", "libsndfile1", "librubberband2"], check=True)

# Pacchetti Python (headless – nessun Qt/GUI)
PACKAGES = [
    # audio / video
    "pydub", "soundfile", "ffmpeg-python", "scipy", "ten-vad",
    # ASR
    "faster-whisper",
    "openai-whisper",
    # TTS
    "edge-tts",
    # subtitles
    "srt",
    "pyrubberband",
    # translation / AI APIs
    "openai",
    "deepl",              # TRANSLATE_TYPE=16 "tenacity", "httpx", "aiohttp", "elevenlabs",
    # utilities
    "requests", "tqdm", "psutil", "zhconv", "jieba", "regex",
    # notebook async compatibility
    "nest_asyncio",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + PACKAGES)
print("✓ Dipendenze installate.")

## Cella 2 – Clona il repository e monta Google Drive

Il codice backend (`videotrans/`) viene scaricato da GitHub.  
Il tuo Google Drive viene montato in `/content/drive` per leggere i video e salvare i risultati.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = "/content/ipynbVideoTrans"

if not Path(REPO_DIR).exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/BaffoBello14/ipynbVideoTrans.git",
         REPO_DIR],
        check=True
    )
    print(f"✓ Repository clonato in {REPO_DIR}")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print(f"✓ Repository aggiornato.")

# Aggiungi il repo al path Python
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Monta Google Drive (opzionale – commenta se non ti serve)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("✓ Google Drive montato in /content/drive")
except Exception:
    print("ℹ  Google Drive non disponibile (ambiente non-Colab o già montato).")

## Cella 3 – Carica il video (scegli un metodo)

Esegui **una sola** delle quattro celle qui sotto in base a dove si trova il tuo video.

In [ ]:
# ── Metodo A: Upload diretto dal tuo computer ──────────────────────────────
# from google.colab import files
# uploaded = files.upload()                         # si apre il selettore file
# INPUT_VIDEO = "/content/" + list(uploaded.keys())[0]
# print(f"✓ Video caricato: {INPUT_VIDEO}")

In [ ]:
# # ── Metodo B: Da Google Drive ──────────────────────────────────────────────
# # Modifica il percorso con la posizione del tuo file su Drive
INPUT_VIDEO = "/content/drive/MyDrive/video.mp4"
print(f"Video da Drive: {INPUT_VIDEO}")

In [ ]:
# # ── Metodo C: Da URL (wget) ────────────────────────────────────────────────
# import subprocess
# VIDEO_URL   = "https://example.com/video.mp4"     # sostituisci con il tuo URL
# INPUT_VIDEO = "/content/video.mp4"
# subprocess.run(["wget", "-q", "-O", INPUT_VIDEO, VIDEO_URL], check=True)
# print(f"✓ Video scaricato: {INPUT_VIDEO}")

In [ ]:
# ── Metodo D: Da Content (path fisso) ──────────────────────────────────────
#INPUT_VIDEO = "/content/video.mp4"
#print(f"✓ Video da Content: {INPUT_VIDEO}")

## Cella 4 – Configurazione

**Modifica qui** tutti i parametri prima di avviare il pipeline.

In [ ]:
import os
from pathlib import Path

# ── Percorsi ──────────────────────────────────────────────────────────────
# INPUT_VIDEO è già definito nella cella precedente.
# OUTPUT_DIR: default locale; per Drive decommenta la riga sotto e adatta il percorso.
# OUTPUT_DIR = "/content/output"               # cartella su Colab (sempre scrivibile)
OUTPUT_DIR = "/content/drive/MyDrive/Folder"  # esempio Google Drive

# ── Lingue ────────────────────────────────────────────────────────────────
SOURCE_LANG   = "it"     # lingua parlata nel video  (zh, en, it, ja, ko, fr, de, es, …)
TARGET_LANG   = "en"     # lingua di destinazione

# ── ASR (riconoscimento vocale) ───────────────────────────────────────────
#   0 = Faster-Whisper  (locale)
#   1 = OpenAI-Whisper  (locale)
#   5 = OpenAI Speech API  (remoto, richiede OPENAI_API_KEY)
#   6 = Gemini AI          (remoto, richiede GEMINI_API_KEY)
RECOGN_TYPE   = 1
WHISPER_MODEL = "large-v3-turbo"   # tiny | small | base | medium | large-v3-turbo | large-v3
USE_CUDA      = True                # False se Runtime solo CPU

# ── Qualità ASR / allineamento doppiaggio (VTV) ───────────────────────────
# remove_noise: preprocessing del segnale prima dell'ASR (utile con strada / Zoom).
REMOVE_NOISE    = False
# fix_punc: dopo la trascrizione, recupera punteggiatura (testo più leggibile / TTS).
FIX_PUNC        = True
# Con autorate attivi, in fase di allineamento la fine di ogni segmento viene estesa
# fino all'inizio del successivo (include il silenzio tra due righe SRT). Vedi
# videotrans/task/_rate.py (docstring in cima).
VOICE_AUTORATE  = True   # accelerazione audio per entrare nello slot temporale
VIDEO_AUTORATE  = True   # rallentamento video nello slot se il TTS è ancora troppo lungo

# ── Silenzi in coda allo slot (solo con VOICE_AUTORATE + pyrubberband) ─────
# Se il doppiaggio è più corto dello slot, allunga leggermente l'audio invece di solo silenzio.
# 0 = disattivato. Richiede apt librubberband2 e pip pyrubberband.
STRETCH_SHORT_MAX_MS    = 400
STRETCH_SHORT_MAX_RATIO = 1.12   # es. 1.12 = al massimo +12% sulla durata del clip

# ── Flusso e diagnostica ─────────────────────────────────────────────────
# True = dopo la traduzione la pipeline si ferma (input in console) per modificare TARGET_SRT.
PAUSE_AFTER_TRANSLATION = False
# True = all'inizio di ogni run viene cancellata tutta OUTPUT_DIR (attenzione alle correzioni manuali).
CLEAR_CACHE_BEFORE_RUN  = False
# True = fine Cella 5: stampa provider ASR/TTS/traduzione + campione voci Edge-TTS.
RUN_DIAGNOSTICS_AT_INIT = False
EDGE_VOICE_LIST_FILTER  = "en-"   # filtro nomi corti Edge (es. "it-", "en-US")

# ── Traduzione ────────────────────────────────────────────────────────────
#   0  = Google Translate   (gratis, nessuna chiave)
#   1  = Microsoft Translate
#   3  = OpenAI ChatGPT     (richiede OPENAI_API_KEY)
#   4  = DeepSeek           (richiede DEEPSEEK_API_KEY)
#   5  = Gemini AI          (richiede GEMINI_API_KEY)
#  16  = DeepL              (richiede DEEPL_API_KEY)
TRANSLATE_TYPE = 16


# ── Qualità traduzione ───────────────────────────────────────────────────────
# Se usi un provider AI (DeepSeek, OpenAI, Gemini…) imposta True per inviare
# l'intero SRT con timestamp all'LLM: il contesto migliora drasticamente la qualità.
# Con Google/Microsoft deve restare False.
AI_SENDS_SRT   = TRANSLATE_TYPE in [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 23, 24]

# Numero di FRASI COMPLETE per batch (solo per provider non-AI).
# Le righe SRT frammentate vengono prima fuse in frasi complete (stile AutoDub),
# poi le frasi vengono raggruppate in batch di questa dimensione prima della chiamata API.
# Valori bassi = richieste HTTP più piccole; valori alti = meno chiamate API.
# Consigliato: 10-20 (frasi, non righe SRT).
TRANS_THREAD   = 20

# Righe SRT per batch quando si usa AI con AI_SENDS_SRT=True.
AITRANS_THREAD = 30

# Se True, invia l'intero testo del video come contesto globale all'AI
# (migliora coerenza terminologica ma usa più token).
AI_GLOBAL_CONTEXT = True

# ── TTS (sintesi vocale) ──────────────────────────────────────────────────
#   0  = Edge-TTS   (gratis, Microsoft, raccomandato)
#  15  = OpenAI TTS (richiede OPENAI_API_KEY)
#  29  = Google TTS (gratis)
TTS_TYPE      = 0
# Voce Edge-TTS: usa la cella Utilità per elencare le voci disponibili
# Esempi: "it-IT-ElsaNeural", "it-IT-DiegoNeural", "en-US-JennyNeural"
VOICE_ROLE    = "en-US-GuyNeural"
VOICE_RATE    = "+0%"    # velocità: "+20%" più veloce, "-20%" più lento
VOLUME        = "+0%"
PITCH         = "+0Hz"

# ── Sottotitoli nel video finale ──────────────────────────────────────────
#   0 = nessun sottotitolo, 1 = hard (bruciati), 2 = soft (selezionabili)
SUBTITLE_TYPE = 1

# ── Chiavi API opzionali ──────────────────────────────────────────────────
# Inseriscile qui oppure usa os.environ / Colab Secrets (chiave a forma di lucchetto)
try:
    from google.colab import userdata as _colab_userdata
except ImportError:
    _colab_userdata = None
def get_key(name):
    try:
        if _colab_userdata:
            val = _colab_userdata.get(name)
            if val: return val
    except Exception:
        pass
    return os.environ.get(name, "")

OPENAI_API_KEY   = get_key("OPENAI_API_KEY")
GEMINI_API_KEY   = get_key("GEMINI_API_KEY")
DEEPSEEK_API_KEY = get_key("DEEPSEEK_API_KEY")
DEEPL_API_KEY    = get_key("DEEPL_API_KEY")

# ── Percorsi derivati (non modificare) ────────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
_stem      = Path(INPUT_VIDEO).stem
SOURCE_SRT = str(Path(OUTPUT_DIR) / f"{SOURCE_LANG}.srt")   # matches trans_create.py naming
TARGET_SRT = str(Path(OUTPUT_DIR) / f"{TARGET_LANG}.srt")   # matches trans_create.py naming
DUBBED_WAV = str(Path(OUTPUT_DIR) / f"{_stem}_dubbed.wav")
OUTPUT_MP4 = str(Path(OUTPUT_DIR) / f"{_stem}_translated.mp4")

print(f"Input  : {INPUT_VIDEO}")
print(f"Output : {OUTPUT_DIR}")
print(f"Lingua : {SOURCE_LANG} → {TARGET_LANG}")
print(f"ASR    : type={RECOGN_TYPE}, model={WHISPER_MODEL}, cuda={USE_CUDA}, remove_noise={REMOVE_NOISE}, fix_punc={FIX_PUNC}")
print(f"Trans  : type={TRANSLATE_TYPE}")
print(f"TTS    : type={TTS_TYPE}, voce={VOICE_ROLE}")
print(f"Align  : voice_autorate={VOICE_AUTORATE}, video_autorate={VIDEO_AUTORATE}")
print(f"Stretch: max_ms={STRETCH_SHORT_MAX_MS}, max_ratio={STRETCH_SHORT_MAX_RATIO} (serve pyrubberband)")
print(f"Pausa dopo trad.: {PAUSE_AFTER_TRANSLATION} | svuota OUTPUT_DIR: {CLEAR_CACHE_BEFORE_RUN} | diag: {RUN_DIAGNOSTICS_AT_INIT}")


## Cella 5 – Inizializza il backend pyVideoTrans

In [ ]:
import sys, os
from pathlib import Path

REPO_DIR = "/content/ipynbVideoTrans"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import subprocess, importlib
# Pull latest fixes from the repository before importing
subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
importlib.invalidate_caches()
# Reload key modules so fixes take effect without kernel restart
for _mod_name in list(__import__('sys').modules.keys()):
    if _mod_name.startswith('videotrans'):
        __import__('sys').modules.pop(_mod_name)



from videotrans.configure import config
config.init_run()

from videotrans.configure.config import ROOT_DIR, TEMP_DIR, app_cfg, settings, params, logger

# Espone le chiavi API al layer dei parametri interni
if OPENAI_API_KEY:
    params['openaitts_key']       = OPENAI_API_KEY
    params['chatgpt_key']         = OPENAI_API_KEY
    params['openairecognapi_key'] = OPENAI_API_KEY
if GEMINI_API_KEY:
    params['gemini_key']          = GEMINI_API_KEY
if DEEPSEEK_API_KEY:
    params['deepseek_key']        = DEEPSEEK_API_KEY
if DEEPL_API_KEY:
    params['deepl_authkey']        = DEEPL_API_KEY  # key used by _deepl.py

app_cfg.exit_soft = False
app_cfg.exec_mode = 'cli'
# Applica le impostazioni di qualità traduzione
settings['trans_thread']     = TRANS_THREAD
settings['aitrans_thread']   = AITRANS_THREAD
settings['aisendsrt']        = AI_SENDS_SRT
settings['aitrans_context']  = AI_GLOBAL_CONTEXT


from videotrans.util import tools
from videotrans.util.gpus import getset_gpu
getset_gpu()

# Verifica CUDA
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"CUDA disponibile: {cuda_ok}" + (f" – {torch.cuda.get_device_name(0)}" if cuda_ok else ""))
    if USE_CUDA and not cuda_ok:
      print("⚠ USE_CUDA=True ma nessuna GPU trovata → imposto USE_CUDA=False automaticamente")
      USE_CUDA = False
except ImportError:
    print("PyTorch non installato.")

try:
    import pyrubberband  # noqa: F401
    print("pyrubberband: OK (allineamento / stretch audio)")
except ImportError:
    print("pyrubberband: non installato — stretch e accelerazione Rubber Band disabilitati")

print(f"ROOT_DIR : {ROOT_DIR}")
print("✓ Backend inizializzato.")

if RUN_DIAGNOSTICS_AT_INIT:
    print("\n--- Diagnostica (RUN_DIAGNOSTICS_AT_INIT) ---")
    try:
        import shutil, subprocess
        ff = shutil.which("ffmpeg")
        print("ffmpeg:", ff or "NON TROVATO")
        if ff:
            v = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
            print(" ", v.stdout.split("\n")[0])
    except Exception as e:
        print("ffmpeg:", e)
    try:
        from videotrans import recognition, translator, tts
        print("ASR:", len(recognition.RECOGN_NAME_LIST), "| Trans:", len(translator.TRANSLASTE_NAME_LIST), "| TTS:", len(tts.TTS_NAME_LIST))
        for i, n in enumerate(recognition.RECOGN_NAME_LIST[:6]):
            print(f"  ASR {i:>2}: {n}")
        print("  ...")
    except Exception as e:
        print("Provider:", e)
    try:
        import asyncio
        import edge_tts
        async def _list_edge(locale_filter):
            voci = await edge_tts.list_voices()
            n = 0
            for v in voci:
                if locale_filter.lower() in v["ShortName"].lower():
                    print(f"  {v['ShortName']:42s}  {v['Gender']}")
                    n += 1
                    if n >= 20:
                        print("  ... (max 20)")
                        break
        print("Edge-TTS (max 20), filtro:", repr(EDGE_VOICE_LIST_FILTER))
        try:
            import nest_asyncio
            nest_asyncio.apply()
        except ImportError:
            pass
        asyncio.get_event_loop().run_until_complete(_list_edge(EDGE_VOICE_LIST_FILTER))
    except Exception as e:
        print("Edge-TTS:", e)


## Cella 6 – Pipeline completa (ASR → Traduzione → TTS → MP4)

Esegui questa cella per tradurre il video in un colpo solo.

In [ ]:
import uuid
from pathlib import Path
from videotrans.task.trans_create import TransCreate
from videotrans.task.taskcfg import TaskCfgVTT

task_uuid    = uuid.uuid4().hex[:10]
cache_folder = str(Path(TEMP_DIR) / task_uuid)
Path(cache_folder).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

file_obj = tools.format_video(Path(INPUT_VIDEO).absolute().as_posix())
file_obj.pop('uuid', None)       # avoid duplicate kwarg
file_obj.pop('target_dir', None)  # avoid duplicate kwarg

vtv_cfg = TaskCfgVTT(
    **file_obj,
    uuid                 = task_uuid,
    cache_folder         = cache_folder,
    target_dir           = OUTPUT_DIR,
    # Lingue
    source_language_code = SOURCE_LANG,
    target_language_code = TARGET_LANG,
    # ASR
    recogn_type          = RECOGN_TYPE,
    model_name           = WHISPER_MODEL,
    is_cuda              = USE_CUDA,
    remove_noise         = REMOVE_NOISE,
    enable_diariz        = False,
    rephrase             = 0,
    fix_punc             = FIX_PUNC,
    # Traduzione
    translate_type       = TRANSLATE_TYPE,
    # TTS
    tts_type             = TTS_TYPE,
    voice_role           = VOICE_ROLE,
    voice_rate           = VOICE_RATE,
    volume               = VOLUME,
    pitch                = PITCH,
    voice_autorate       = VOICE_AUTORATE,
    video_autorate       = VIDEO_AUTORATE,
    stretch_short_max_ms    = STRETCH_SHORT_MAX_MS,
    stretch_short_max_ratio = STRETCH_SHORT_MAX_RATIO,
    align_sub_audio      = True,
    # Output
    subtitle_type        = SUBTITLE_TYPE,
    is_separate          = False,
    recogn2pass          = False,
    clear_cache          = CLEAR_CACHE_BEFORE_RUN,
)

app_cfg.current_status = 'ing'
print("Avvio pipeline VTV completa…")

trk = TransCreate(cfg=vtv_cfg)
trk.prepare()
trk.recogn()
trk.trans()
if PAUSE_AFTER_TRANSLATION:
    _ts = Path(TARGET_SRT)
    print("\n" + "=" * 72)
    print("PAUSA: correggi la traduzione in questo file, salva, poi premi Invio qui sotto.")
    print(_ts.resolve() if _ts.exists() else TARGET_SRT)
    print("=" * 72 + "\n")
    input("[Invio quando TARGET_SRT è salvato] ")
trk.dubbing()
trk.align()
trk.assembling()
trk.task_done()

print(f"\n✓ Completato! I file sono in: {OUTPUT_DIR}")
import os
for f in os.listdir(OUTPUT_DIR):
    print(f"  {f}")


## Cella 7 – Scarica il risultato

Scarica il MP4 finale sul tuo computer (oppure salvalo su Drive se `OUTPUT_DIR` punta lì).

In [ ]:
import os
from pathlib import Path

# Cerca il file MP4 prodotto
mp4_files = list(Path(OUTPUT_DIR).glob("*.mp4"))

if mp4_files:
    try:
        from google.colab import files
        for mp4 in mp4_files:
            print(f"Download: {mp4}")
            files.download(str(mp4))
    except ImportError:
        print("File disponibili in:", OUTPUT_DIR)
        for f in mp4_files:
            print(f"  {f}")
else:
    print(f"Nessun MP4 trovato in {OUTPUT_DIR}. Controlla i log.")